<a href="https://colab.research.google.com/github/Kshama-Hiremath/rgb-to-grayscale-cuda/blob/main/PC_CCA2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!nvidia-smi

Fri Oct  9 06:55:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [5]:
%%writefile rgb_to_grayscale.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N 10

struct RGB {
    unsigned char r, g, b;
};

__global__ void rgbToGrayscale(RGB *input,
                               unsigned char *output, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n) {
        float gray = 0.299f * input[i].r
                   + 0.587f * input[i].g
                   + 0.114f * input[i].b;

        output[i] = (unsigned char)(gray + 0.5f);
    }
}

int main() {
    RGB h_input[N] = {
        {255,0,0}, {0,255,0}, {0,0,255},
        {255,255,255}, {0,0,0}, {255,255,0},
        {255,0,255}, {0,255,255}, {128,128,128},
        {100,150,200}
    };

    unsigned char h_output[N];
    RGB *d_input;
    unsigned char *d_output;

    cudaMalloc((void**)&d_input, N * sizeof(RGB));
    cudaMalloc((void**)&d_output, N);

    cudaMemcpy(d_input, h_input, N * sizeof(RGB),
               cudaMemcpyHostToDevice);

    int threads = 256;
    int blocks = (N + threads - 1) / threads;

    rgbToGrayscale<<<blocks, threads>>>(d_input, d_output, N);
    cudaDeviceSynchronize();

    cudaMemcpy(h_output, d_output, N,
               cudaMemcpyDeviceToHost);

    printf("RGB to Grayscale using CUDA\n");
    printf("----------------------------\n");

    for (int i = 0; i < N; i++) {
        printf("Pixel %d: RGB(%d,%d,%d) -> Grayscale %d\n",
               i, h_input[i].r, h_input[i].g,
               h_input[i].b, h_output[i]);
    }

    cudaFree(d_input);
    cudaFree(d_output);
    return 0;
}

Writing rgb_to_grayscale.cu


In [9]:
%%writefile rgb_benchmark.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <chrono>
#include <cmath>

#define N 1000000

struct RGB {
    unsigned char r, g, b;
};

__global__ void grayscaleGPU(RGB *in, unsigned char *out, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n) {
        float gray = 0.299f * in[i].r
                   + 0.587f * in[i].g
                   + 0.114f * in[i].b;

        out[i] = (unsigned char)(gray + 0.5f);
    }
}

void grayscaleCPU(RGB *in, unsigned char *out, int n) {
    for (int i = 0; i < n; i++) {
        float gray = 0.299f * in[i].r
                   + 0.587f * in[i].g
                   + 0.114f * in[i].b;

        out[i] = (unsigned char)(gray + 0.5f);
    }
}

int main() {
    RGB *h_in = (RGB*)malloc(N * sizeof(RGB));
    unsigned char *cpu_out =
        (unsigned char*)malloc(N);
    unsigned char *gpu_out =
        (unsigned char*)malloc(N);

    for (int i = 0; i < N; i++) {
        h_in[i].r = (i * 17) % 256;
        h_in[i].g = (i * 31 + 40) % 256;
        h_in[i].b = (i * 13 + 90) % 256;
    }

    auto c1 = std::chrono::high_resolution_clock::now();
    grayscaleCPU(h_in, cpu_out, N);
    auto c2 = std::chrono::high_resolution_clock::now();

    double cpu_ms =
        std::chrono::duration<double, std::milli>(c2-c1).count();

    RGB *d_in;
    unsigned char *d_out;

    cudaMalloc((void**)&d_in, N * sizeof(RGB));
    cudaMalloc((void**)&d_out, N);

    auto g1 = std::chrono::high_resolution_clock::now();

    cudaMemcpy(d_in, h_in, N * sizeof(RGB),
               cudaMemcpyHostToDevice);

    int threads = 256;
    int blocks = (N + threads - 1) / threads;

    grayscaleGPU<<<blocks, threads>>>(d_in, d_out, N);
    cudaDeviceSynchronize();

    cudaMemcpy(gpu_out, d_out, N,
               cudaMemcpyDeviceToHost);

    auto g2 = std::chrono::high_resolution_clock::now();

    double gpu_ms =
        std::chrono::duration<double, std::milli>(g2-g1).count();

    int errors = 0;
    for (int i = 0; i < N; i++) {
        if (abs((int)cpu_out[i] - (int)gpu_out[i]) > 1)
            errors++;
    }

    printf("\nRGB TO GRAYSCALE: CPU vs GPU\n");
    printf("----------------------------\n");
    printf("Number of pixels       : %d\n", N);
    printf("CPU execution time     : %.4f ms\n", cpu_ms);
    printf("GPU total execution    : %.4f ms\n", gpu_ms);
    printf("End-to-end speedup     : %.2fx\n", cpu_ms / gpu_ms);
    printf("Mismatched pixels      : %d\n", errors);
    printf("Correctness            : %s\n",
           errors == 0 ? "PASS" : "FAIL");

    cudaFree(d_in);
    cudaFree(d_out);
    free(h_in);
    free(cpu_out);
    free(gpu_out);

    return errors == 0 ? 0 : 1;
}

Writing rgb_benchmark.cu


In [11]:
!sed 's/#define N 1000000/#define N 10000/' rgb_benchmark.cu > benchmark_10k.cu
!nvcc -O2 benchmark_10k.cu -o benchmark_10k
!./benchmark_10k


RGB TO GRAYSCALE: CPU vs GPU
----------------------------
Number of pixels       : 10000
CPU execution time     : 0.0569 ms
GPU total execution    : 0.2060 ms
End-to-end speedup     : 0.28x
Mismatched pixels      : 0
Correctness            : PASS


In [12]:
!sed 's/#define N 1000000/#define N 100000/' rgb_benchmark.cu > benchmark_100k.cu
!nvcc -O2 benchmark_100k.cu -o benchmark_100k
!./benchmark_100k


RGB TO GRAYSCALE: CPU vs GPU
----------------------------
Number of pixels       : 100000
CPU execution time     : 0.3690 ms
GPU total execution    : 0.2869 ms
End-to-end speedup     : 1.29x
Mismatched pixels      : 0
Correctness            : PASS


In [13]:
!sed 's/#define N 1000000/#define N 10000000/' rgb_benchmark.cu > benchmark_10m.cu
!nvcc -O2 benchmark_10m.cu -o benchmark_10m
!./benchmark_10m


RGB TO GRAYSCALE: CPU vs GPU
----------------------------
Number of pixels       : 10000000
CPU execution time     : 25.2972 ms
GPU total execution    : 13.5599 ms
End-to-end speedup     : 1.87x
Mismatched pixels      : 0
Correctness            : PASS


In [10]:
!nvcc -O2 rgb_benchmark.cu -o rgb_benchmark
!./rgb_benchmark


RGB TO GRAYSCALE: CPU vs GPU
----------------------------
Number of pixels       : 1000000
CPU execution time     : 2.7814 ms
GPU total execution    : 1.6536 ms
End-to-end speedup     : 1.68x
Mismatched pixels      : 0
Correctness            : PASS


In [6]:
!nvcc rgb_to_grayscale.cu -o grayscale
!./grayscale

RGB to Grayscale using CUDA
----------------------------
Pixel 0: RGB(255,0,0) -> Grayscale 76
Pixel 1: RGB(0,255,0) -> Grayscale 150
Pixel 2: RGB(0,0,255) -> Grayscale 29
Pixel 3: RGB(255,255,255) -> Grayscale 255
Pixel 4: RGB(0,0,0) -> Grayscale 0
Pixel 5: RGB(255,255,0) -> Grayscale 226
Pixel 6: RGB(255,0,255) -> Grayscale 105
Pixel 7: RGB(0,255,255) -> Grayscale 179
Pixel 8: RGB(128,128,128) -> Grayscale 128
Pixel 9: RGB(100,150,200) -> Grayscale 141


In [7]:
!ls -l rgb_to_grayscale.cu

-rw-r--r-- 1 root root 1511 Oct  9 06:57 rgb_to_grayscale.cu


In [8]:
!nvidia-smi

Fri Oct  9 07:00:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----